In [ ]:
#@title ⚙️ Sel 1: Mengunduh Engine & Install Library { display-mode: "form" }
import os
import subprocess

print("📥 Mengunduh repositori Mapperatorinator dari GitHub...")
subprocess.run(["git", "clone", "https://github.com/OliBomby/Mapperatorinator.git"], check=True)

# Berpindah ke folder Mapperatorinator
os.chdir("/content/Mapperatorinator")
print("📁 Posisi saat ini:", os.getcwd())

print("📦 Menginstal library bawaan...")
subprocess.run(["pip", "install", "-r", "requirements.txt"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pip", "install", "-e", "."], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("📦 Menginstal library khusus Training LoRA...")
subprocess.run(["pip", "install", "-U", "peft", "accelerate", "bitsandbytes", "transformers", "datasets"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✅ Sel 1 Selesai! Mesin sudah siap. Lanjut ke Sel 2.")

In [ ]:
#@title 📁 Sel 2: Sambungkan ke Google Drive & Buat Folder { display-mode: "form" }
import os
from google.colab import drive

print("Membuka koneksi ke Google Drive...")
# Akan muncul pop-up meminta izin akses Google Drive, silakan izinkan
drive.mount('/content/drive')

# Menentukan letak folder di Google Drive
base_dir = '/content/drive/MyDrive/Mapperatorinator_Training'
dataset_dir = os.path.join(base_dir, 'dataset_osu')
output_dir = os.path.join(base_dir, 'lora_output')

# Membuat folder secara otomatis
os.makedirs(dataset_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print("\n✅ WORKSPACE SIAP!")
print(f"📁 Folder Dataset: {dataset_dir}")
print(f"📁 Folder Output (Hasil LoRA): {output_dir}")

In [ ]:
#@title 🚀 Sel 3: Konfigurasi & Mulai Training LoRA { display-mode: "form" }

#@markdown #### Pengaturan Identitas LoRA
nama_mapper = "Gaya_Sotarks" # @param {type:"string"}
#@markdown #### Pengaturan Training
epochs = 10 # @param {type:"slider", min:1, max:50, step:1}
batch_size = 4 # @param {type:"slider", min:1, max:16, step:1}

import os
import subprocess

# Memastikan Colab berada di folder utama engine
os.chdir("/content/Mapperatorinator")

dataset_dir = '/content/drive/MyDrive/Mapperatorinator_Training/dataset_osu'
output_dir = f'/content/drive/MyDrive/Mapperatorinator_Training/lora_output/{nama_mapper}'
os.makedirs(output_dir, exist_ok=True)

# Menghitung jumlah file .osu di Google Drive
jumlah_file = len([f for f in os.listdir(dataset_dir) if f.endswith('.osu')])

if jumlah_file == 0:
    print("❌ ERROR: Folder dataset_osu masih KOSONG!")
    print("Silakan masukkan file .osu ke Google Drive kamu terlebih dahulu sebelum memulai training.")
else:
    print(f"✅ Ditemukan {jumlah_file} file .osu. Siap untuk training!")
    print(f"🔥 Memulai proses training untuk {nama_mapper} selama {epochs} epochs...")
    print("⏳ Proses ini memakan waktu yang cukup lama. Jangan tutup tab browser ini!\n")

    try:
        # Perintah training
        training_command = [
            "python", "train.py",
            f"model.peft_config.r=16",
            f"model.peft_config.lora_alpha=32",
            f"dataset.path={dataset_dir}",
            f"training.epochs={str(epochs)}",
            f"training.batch_size={str(batch_size)}",
            f"training.output_dir={output_dir}"
        ]

        # Eksekusi training
        # Menghapus komentar (tanda pagar #) di bawah ini akan benar-benar menjalankan script train.py
        subprocess.run(training_command, check=True)

        print(f"\n🎉 TRAINING SELESAI!")
        print(f"File LoRA kamu tersimpan di Google Drive: {output_dir}")

    except Exception as e:
        print(f"❌ Terjadi kesalahan saat proses training: {e}")